In [2]:
# This script:
# 1. Loads the saved train/test arrays from Step 4
# 2. Tunes XGBoost with RandomizedSearchCV (primary model)
# 3. Tunes Logistic Regression with RandomizedSearchCV
# 4. Tunes Random Forest with RandomizedSearchCV (lighter search)
# 5. Compares before/after tuning for all three models
# 6. Saves the best models for Step 6 (Evaluation & Explainability)
 

import pandas as pd
import numpy as np
import os
import warnings
import joblib
warnings.filterwarnings("ignore")

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import roc_auc_score, average_precision_score, f1_score
import xgboost as xgb

In [3]:
OUTPUT_DIR   = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs"
RANDOM_STATE = 42
 
X_train = np.load(os.path.join(OUTPUT_DIR, "X_train.npy"))
X_test  = np.load(os.path.join(OUTPUT_DIR, "X_test.npy"))
y_train = np.load(os.path.join(OUTPUT_DIR, "y_train.npy"))
y_test  = np.load(os.path.join(OUTPUT_DIR, "y_test.npy"))
 
feature_cols = pd.read_csv(os.path.join(OUTPUT_DIR, "feature_names.csv")).iloc[:, 0].tolist()
 
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Features: {len(feature_cols)}")
 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
 

X_train: (42029, 44), X_test: (10508, 44)
Features: 44


In [4]:
def evaluate(model, X_test, y_test, label=""):
    """Quick evaluation helper — returns dict of test metrics."""
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    metrics = {
        "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4),
        "PR-AUC":  round(average_precision_score(y_test, y_prob), 4),
        "F1":      round(f1_score(y_test, y_pred), 4),
    }
    if label:
        print(f"  {label}: ROC-AUC={metrics['ROC-AUC']} | PR-AUC={metrics['PR-AUC']} | F1={metrics['F1']}")
    return metrics
 
# Load baseline models from Step 4 for before/after comparison
baseline_scores = {}
for name, fname in [("Logistic Regression", "logistic_regression_base.pkl"),
                    ("Random Forest",        "random_forest_base.pkl"),
                    ("XGBoost",              "xgboost_base.pkl")]:
    model = joblib.load(os.path.join(OUTPUT_DIR, fname))
    baseline_scores[name] = evaluate(model, X_test, y_test, label=f"{name} (baseline)")
 

  Logistic Regression (baseline): ROC-AUC=0.7011 | PR-AUC=0.7364 | F1=0.669
  Random Forest (baseline): ROC-AUC=0.6867 | PR-AUC=0.7129 | F1=0.6634
  XGBoost (baseline): ROC-AUC=0.7251 | PR-AUC=0.7585 | F1=0.7097


In [5]:
# 1. Tune XGBoost

# Search space covers the most impactful XGBoost hyperparameters:
#   n_estimators    — number of trees; more trees = stronger but slower
#   max_depth       — tree depth; deeper = more complex, higher overfitting risk
#   learning_rate   — shrinkage; lower = better generalisation but needs more trees
#   subsample       — row sampling per tree; reduces overfitting
#   colsample_bytree— feature sampling per tree; reduces overfitting
#   min_child_weight— minimum sum of weights in a leaf; controls tree growth
#   gamma           — minimum loss reduction for a split; regularisation
#
#50 iterations × 5-fold CV = 250 fits. Scoring:ROC-AUC.
 
print("\nTuning XGBoost ...")
 
xgb_param_dist = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [3, 4, 5, 6, 7],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma":            [0, 0.1, 0.2, 0.5],
}
 
xgb_base = xgb.XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    verbosity=0,
    use_label_encoder=False,
    n_jobs=-1,
)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=50,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
xgb_search.fit(X_train, y_train)
 
print(f"\nBest XGBoost params:\n{xgb_search.best_params_}")
print(f"Best CV ROC-AUC: {xgb_search.best_score_:.4f}")
xgb_tuned_scores = evaluate(xgb_search.best_estimator_, X_test, y_test, label="XGBoost (tuned)")
 
joblib.dump(xgb_search.best_estimator_, os.path.join(OUTPUT_DIR, "xgboost_tuned.pkl"))
 


Tuning XGBoost ...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best XGBoost params:
{'subsample': 1.0, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.6}
Best CV ROC-AUC: 0.7333
  XGBoost (tuned): ROC-AUC=0.7306 | PR-AUC=0.7656 | F1=0.7152


['C:\\Users\\USER\\PycharmProjects\\Non-med-dementia-risk\\outputs\\xgboost_tuned.pkl']

In [6]:
# 2. Tune Logistic Regression
#
# Search space:
#   C          — inverse regularisation strength; smaller = stronger regularisation
#   penalty    — L1 (sparse, feature selection) vs L2 (shrinkage)
#   solver     — must match penalty: saga supports both L1 and L2
#
# 30 iterations × 5-fold CV = 150 fits.

print("\nTuning Logistic Regression ...")
 
lr_param_dist = {
    "clf__C":       [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
    "clf__penalty": ["l1", "l2"],
    "clf__solver":  ["saga"],       # saga supports both l1 and l2
}
 
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(
                   max_iter=2000,
                   random_state=RANDOM_STATE,
                   class_weight="balanced"
               ))
])

lr_search = RandomizedSearchCV(
    lr_pipeline,
    param_distributions=lr_param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
lr_search.fit(X_train, y_train)
 
print(f"\nBest LR params:\n{lr_search.best_params_}")
print(f"Best CV ROC-AUC: {lr_search.best_score_:.4f}")
lr_tuned_scores = evaluate(lr_search.best_estimator_, X_test, y_test, label="Logistic Regression (tuned)")
 
joblib.dump(lr_search.best_estimator_, os.path.join(OUTPUT_DIR, "logistic_regression_tuned.pkl"))



Tuning Logistic Regression ...
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Best LR params:
{'clf__solver': 'saga', 'clf__penalty': 'l1', 'clf__C': 0.1}
Best CV ROC-AUC: 0.7049
  Logistic Regression (tuned): ROC-AUC=0.7011 | PR-AUC=0.7363 | F1=0.6691


['C:\\Users\\USER\\PycharmProjects\\Non-med-dementia-risk\\outputs\\logistic_regression_tuned.pkl']

In [7]:
# 3. Tune Random Forest
#
# Search space:
#   n_estimators   — number of trees
#   max_depth      — tree depth (None = fully grown)
#   min_samples_leaf — minimum samples per leaf; higher = smoother, less overfit
#   max_features   — features per split; "sqrt" is standard for classification
#
# 30 iterations × 5-fold CV = 150 fits.
 
print("\nTuning Random Forest ...")
 
rf_param_dist = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [None, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 4, 8, 16],
    "max_features":     ["sqrt", "log2", 0.3, 0.5],
}
 
rf_base = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced",
)
 
rf_search = RandomizedSearchCV(
    rf_base,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

rf_search.fit(X_train, y_train)
 
print(f"\nBest RF params:\n{rf_search.best_params_}")
print(f"Best CV ROC-AUC: {rf_search.best_score_:.4f}")
rf_tuned_scores = evaluate(rf_search.best_estimator_, X_test, y_test, label="Random Forest (tuned)")
 
joblib.dump(rf_search.best_estimator_, os.path.join(OUTPUT_DIR, "random_forest_tuned.pkl"))


Tuning Random Forest ...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best RF params:
{'n_estimators': 200, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 20}
Best CV ROC-AUC: 0.7296
  Random Forest (tuned): ROC-AUC=0.7256 | PR-AUC=0.7617 | F1=0.6859


['C:\\Users\\USER\\PycharmProjects\\Non-med-dementia-risk\\outputs\\random_forest_tuned.pkl']

In [8]:
# 4. Before / after comparison

tuned_scores = {
    "Logistic Regression": lr_tuned_scores,
    "Random Forest":        rf_tuned_scores,
    "XGBoost":              xgb_tuned_scores,
}
 
rows = []
for name in ["Logistic Regression", "Random Forest", "XGBoost"]:
    b = baseline_scores[name]
    t = tuned_scores[name]
    rows.append({
        "Model":             name,
        "ROC-AUC (base)":    b["ROC-AUC"],
        "ROC-AUC (tuned)":   t["ROC-AUC"],
        "ROC-AUC (delta)":   round(t["ROC-AUC"] - b["ROC-AUC"], 4),
        "PR-AUC  (base)":    b["PR-AUC"],
        "PR-AUC  (tuned)":   t["PR-AUC"],
        "F1      (base)":    b["F1"],
        "F1      (tuned)":   t["F1"],
    })
    
    comparison = pd.DataFrame(rows).set_index("Model")
print("\n=== Before / After Tuning ===")
print(comparison.to_string())
 
comparison.to_csv(os.path.join(OUTPUT_DIR, "step5_tuning_comparison.csv"))
print(f"\nSaved: step5_tuning_comparison.csv")
print(f"Best model: XGBoost (tuned) — ROC-AUC {xgb_tuned_scores['ROC-AUC']} on test set")
 


=== Before / After Tuning ===
                     ROC-AUC (base)  ROC-AUC (tuned)  ROC-AUC (delta)  PR-AUC  (base)  PR-AUC  (tuned)  F1      (base)  F1      (tuned)
Model                                                                                                                                  
Logistic Regression          0.7011           0.7011           0.0000          0.7364           0.7363          0.6690           0.6691
Random Forest                0.6867           0.7256           0.0389          0.7129           0.7617          0.6634           0.6859
XGBoost                      0.7251           0.7306           0.0055          0.7585           0.7656          0.7097           0.7152

Saved: step5_tuning_comparison.csv
Best model: XGBoost (tuned) — ROC-AUC 0.7306 on test set
